# Дополнительное задание. Оценка константы Липшица через вторую производную

**Задание:** реализовать поиск константы Липшица с использованием второй производной и объяснить принцип.

## Теоретическое обоснование

Для дифференцируемой функции на отрезке [a,b] константа Липшица может быть выбрана как


$L=\max_{x\in[a,b]}|f'(x)|$,


поскольку по теореме Лагранжа для любых (u,v) существует точка c между ними, и

$\frac{|f(u)-f(v)|}{|u-v|}=|f'(c)|\leq L$

Если напрямую найти максимум модуля первой производной сложно, используем вторую производную. Пусть на отрезке известно (или численно оценено) ограничение
$
|f''(x)|\leq M_2.
$
По теореме Лагранжа для \(f'\):
$
|f'(x)-f'(m)|\leq M_2|x-m|,
$
где (m) — середина небольшого подотрезка. Следовательно, для любого (x) внутри подотрезка длины (h):
$
|f'(x)|\leq |f'(m)|+M_2\frac h2.
$

Разбиваем исходный отрезок на подотрезки, в каждом вычисляем правую часть и берём максимум:
$
L_{est}=\max_i\left(|f'(m_i)|+M_{2,i}\frac{h_i}{2}\right).
$
Затем добавляем небольшой коэффициент запаса $(r\geq1)$: $(L=rL_{est})$.

**Смысл метода:** вторая производная ограничивает, насколько быстро меняется первая производная. Поэтому значение |f'| в середине ячейки плюс возможное изменение на половине ячейки даёт оценку сверху для наклона функции в этой ячейке.

> Важно: если M_2 получен лишь выборкой значений на сетке, результат является численной оценкой, а не строгой математической гарантией. Для строгой гарантии нужно доказать верхнюю границу (|f''|
 на каждой ячейке (например, аналитически или средствами интервальной арифметики).

## Реализация на Python
Функция принимает строковое выражение в формате, используемом в основной лабораторной работе, например `f(x) = x + sin(3.14159*x)`. SymPy вычисляет производные аналитически, а сетка используется для численной оценки максимума второй производной.

In [ ]:
import numpy as np
import sympy as sp

def find_L_second_derivative(func_str, a, b, n=2001, reserve=1.2):
    """Численная оценка константы Липшица через f' и ограничение |f''|.

    n — число узлов разбиения; reserve — коэффициент запаса (>= 1).
    """
    if not (np.isfinite(a) and np.isfinite(b)) or a >= b:
        raise ValueError("Требуется конечный отрезок с условием a < b")
    if not isinstance(n, (int, np.integer)) or n < 2:
        raise ValueError("n должно быть целым числом >= 2")
    if not np.isfinite(reserve) or reserve < 1:
        raise ValueError("reserve должно быть конечным и не меньше 1")

    # Выражение после знака равенства; поддерживаются ^ и стандартные функции SymPy.
    expr_text = func_str.split("=", 1)[-1].replace("^", "**").strip()
    x = sp.Symbol("x", real=True)
    expr = sp.sympify(expr_text, locals={
        "x": x, "pi": sp.pi, "e": sp.E,
        "sin": sp.sin, "cos": sp.cos, "tan": sp.tan,
        "exp": sp.exp, "log": sp.log, "sqrt": sp.sqrt,
        "abs": sp.Abs
    })
    f1 = sp.lambdify(x, sp.diff(expr, x), modules="numpy")
    f2 = sp.lambdify(x, sp.diff(expr, x, 2), modules="numpy")

    edges = np.linspace(a, b, n)
    mids = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)

    d1 = np.asarray(f1(mids), dtype=float)
    d2 = np.asarray(f2(mids), dtype=float)
    if not (np.all(np.isfinite(d1)) and np.all(np.isfinite(d2))):
        raise ValueError("Производные имеют нечисловые значения на отрезке")

    # Численная оценка локальной границы |f''| на каждой ячейке.
    # Запас применяется к оценке, чтобы снизить риск недооценки.
    M2 = reserve * np.abs(d2)
    local_bounds = np.abs(d1) + M2 * widths / 2
    L_est = float(np.max(local_bounds))
    return max(L_est * reserve, np.finfo(float).eps)


## Проверка на функции из лабораторной работы
Для `f(x)=x+sin(3.14159*x)` на отрезке [-2, 2] в основной работе получалась оценка около 4.97 методом конечных разностей. Ниже вычисляем оценку с использованием второй производной.

In [ ]:
L2 = find_L_second_derivative("f(x) = x + sin(3.14159*x)", -2, 2)
print(f"Оценка L через вторую производную: {L2:.6f}")

Оценка L через вторую производную: 4.969934


## Итог

В дополнительной реализации производные вычисляются символически. На каждой ячейке отрезка используется оценка
$
|f'(m_i)|+M_{2,i}h_i/2,
$
которая учитывает возможное изменение первой производной внутри ячейки. Максимум этих значений и коэффициент запаса дают оценку константы Липшица, пригодную для передачи в метод ломаных.

Для гарантированно корректной работы метода Пиявского важно не занижать (L). При практическом использовании следует увеличивать число ячеек и коэффициент запаса, а для строгого доказательства — отдельно подтвердить верхнюю границу второй производной.